# B1 — Core analysis

Event-window construction, the difference-in-differences results, and the
robustness work that established firm size as the dominant confound.

**Requires A1 to have been run.** This notebook only reads the intermediate
files; it never rebuilds them. If a pipeline script changes, re-run the
corresponding step in A1 before running this.

B2 continues from here with the period analysis, the 2015 classification
break, and the second uncertainty measure.

## Setup

In [1]:
import sys
from pathlib import Path

def find_src_dir(start: Path = None) -> Path:
    current = start or Path.cwd()
    for _ in range(5):
        candidate = current / "src"
        if candidate.exists():
            return candidate
        current = current.parent
    raise FileNotFoundError("Could not locate a 'src' folder above the current directory.")

SRC_DIR = find_src_dir()
sys.path.append(str(SRC_DIR))

import polars as pl
import numpy as np
import statsmodels.formula.api as smf
from paths import DATA_DIR, IBES_DIR

## Event window selection

The near-event window was chosen empirically rather than by convention.
Retail volume is flat until day −2, spikes at day 0 and +1 to roughly 2.5x
baseline, and returns to baseline by day +5. The window used throughout is
therefore −1 to +4 trading days.

In [2]:
from analysis.event_window_profile import build_event_profile

profile = build_event_profile(window=30)
with pl.Config(tbl_rows=-1):
    print(profile)

Matched 97,270 of 163,010 events to CBOE trading data
shape: (61, 3)
┌─────────┬─────────────────┬──────────┐
│ rel_day ┆ mean_retail_vol ┆ n_events │
│ ---     ┆ ---             ┆ ---      │
│ i64     ┆ f64             ┆ u32      │
╞═════════╪═════════════════╪══════════╡
│ -30     ┆ 705.993012      ┆ 94024    │
│ -29     ┆ 733.257719      ┆ 94122    │
│ -28     ┆ 730.185958      ┆ 94220    │
│ -27     ┆ 732.231435      ┆ 94342    │
│ -26     ┆ 722.294777      ┆ 94468    │
│ -25     ┆ 706.142591      ┆ 94606    │
│ -24     ┆ 720.85865       ┆ 94715    │
│ -23     ┆ 718.836406      ┆ 94820    │
│ -22     ┆ 706.241305      ┆ 94967    │
│ -21     ┆ 699.715598      ┆ 95126    │
│ -20     ┆ 681.803054      ┆ 95280    │
│ -19     ┆ 688.383603      ┆ 95385    │
│ -18     ┆ 694.000325      ┆ 95489    │
│ -17     ┆ 690.638102      ┆ 95654    │
│ -16     ┆ 680.298978      ┆ 95803    │
│ -15     ┆ 663.394114      ┆ 95962    │
│ -14     ┆ 681.641999      ┆ 96092    │
│ -13     ┆ 701.835271      ┆

### Sample representativeness

Confirms the matched sample is not concentrated in the 2020–21 retail boom.

In [3]:
import random

events = pl.read_parquet(IBES_DIR / "dispersion_events.parquet")
random.seed(42)
n_sample = 1000
if events.height > n_sample:
    idx = random.sample(range(events.height), n_sample)
    events_sample = events[idx]

year_counts = (
    events_sample.with_columns(pl.col("ANNDATS_ACT").dt.year().alias("year"))
    .group_by("year").len().sort("year")
)
with pl.Config(tbl_rows=-1):
    print(year_counts)

shape: (12, 2)
┌──────┬─────┐
│ year ┆ len │
│ ---  ┆ --- │
│ i32  ┆ u32 │
╞══════╪═════╡
│ 2011 ┆ 65  │
│ 2012 ┆ 81  │
│ 2013 ┆ 82  │
│ 2014 ┆ 103 │
│ 2015 ┆ 80  │
│ 2016 ┆ 97  │
│ 2017 ┆ 86  │
│ 2018 ┆ 81  │
│ 2019 ┆ 95  │
│ 2020 ┆ 82  │
│ 2021 ┆ 84  │
│ 2022 ┆ 64  │
└──────┴─────┘


## Composition: retail vs. professional customers

Position size and call/put split, near-event against baseline, for both
participant groups.

In [4]:
from analysis.event_window_profile import build_composition_comparison

comparison = build_composition_comparison()
comparison

Matched 97,270 of 163,010 events to CBOE trading data


participant_group,is_near_event,n_obs,share_lt_100,share_100_199,share_gt_199,share_call,share_put
str,bool,u32,f64,f64,f64,f64,f64
"""procust""",false,5230527,0.628166,0.057658,0.314175,0.58493,0.41507
"""procust""",true,581798,0.654998,0.063086,0.281915,0.574271,0.425729
"""retail""",false,5230527,0.705345,0.090275,0.20438,0.646288,0.353712
"""retail""",true,581798,0.719527,0.087516,0.192957,0.624221,0.375779


## Difference-in-differences across all outcomes

`treat:post` answers "does retail shift differently from professional
customers", not merely "does retail shift". The distinction matters: several
outcomes here are driven by professionals moving rather than retail.

In [5]:
from analysis.event_window_profile import build_diff_in_diff_panel, run_diff_in_diff

outcomes = ["lt_100", "call", "otm", "otm_call", "otm_put", "itm", "open"]

results = []
for oc in outcomes:
    panel = build_diff_in_diff_panel(outcome=oc)
    m_ev = run_diff_in_diff(panel, cluster_by="event")
    m_tk = run_diff_in_diff(panel, cluster_by="ticker")
    results.append({
        "outcome": oc,
        "did_coef": m_ev.params["treat:post"],
        "p_event_clustered": m_ev.pvalues["treat:post"],
        "p_ticker_clustered": m_tk.pvalues["treat:post"],
        "n_obs": int(m_ev.nobs),
    })

results_df = pl.DataFrame(results)
with pl.Config(tbl_rows=-1):
    print(results_df)

Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
shape: (7, 5)
┌──────────┬───────────┬───────────────────┬────────────────────┬────────┐
│ outcome  ┆ did_coef  ┆ p_event_clustered ┆ p_ticker_clustered ┆ n_obs  │
│ ---      ┆ ---       ┆

### Levels behind the coefficients

A coefficient cannot show which group moved. Note the persistent baseline
gap: retail sits well above professionals on OTM share even outside event
windows.

In [6]:
for oc in ["otm", "otm_put", "lt_100"]:
    panel = build_diff_in_diff_panel(outcome=oc)
    levels = (
        panel.group_by(["participant_group", "is_near_event"])
        .agg(pl.col("share").mean().alias("mean_share"), pl.len().alias("n"))
        .sort("participant_group", "is_near_event")
    )
    print(f"--- {oc} ---")
    print(levels)
    print()

Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
--- otm ---
shape: (4, 4)
┌───────────────────┬───────────────┬────────────┬───────┐
│ participant_group ┆ is_near_event ┆ mean_share ┆ n     │
│ ---               ┆ ---           ┆ ---        ┆ ---   │
│ str               ┆ bool          ┆ f64        ┆ u32   │
╞═══════════════════╪═══════════════╪════════════╪═══════╡
│ procust           ┆ false         ┆ 0.541283   ┆ 74481 │
│ procust           ┆ true          ┆ 0.542961   ┆ 48325 │
│ retail            ┆ false         ┆ 0.594197   ┆ 97197 │
│ retail            ┆ true          ┆ 0.606598   ┆ 97187 │
└───────────────────┴───────────────┴────────────┴───────┘

Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
--- otm_put ---
shape: (4, 4)
┌───────────────────┬───────────────┬────────────┬───────┐
│ participant_group ┆ is_near_event ┆ mean_share ┆ n     │
│ -

### Moneyness decomposition

The combined OTM effect decomposes almost entirely into puts, with calls
showing nothing. This runs against the lottery-preference literature's
emphasis on out-of-the-money calls, and corroborates the separate call-share
result — two independent measures pointing the same direction.

In [7]:
for oc in ["otm", "otm_call", "otm_put"]:
    panel = build_diff_in_diff_panel(outcome=oc)
    m = run_diff_in_diff(panel, cluster_by="ticker")
    print(f"{oc:9s} treat:post = {m.params['treat:post']:+.4f}, p = {m.pvalues['treat:post']:.4f}")

Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
otm       treat:post = +0.0107, p = 0.0000
Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
otm_call  treat:post = +0.0010, p = 0.6043
Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
otm_put   treat:post = +0.0097, p = 0.0000


## DV3 — opening vs. closing positions

Levels are reported alongside the coefficient because this result is driven
by professionals reducing new position-opening rather than by any change in
retail behaviour.

In [8]:
panel_open = build_diff_in_diff_panel(outcome="open")

m1 = run_diff_in_diff(panel_open, cluster_by="event")
m2 = run_diff_in_diff(panel_open, cluster_by="ticker")
print(f"[open, event-clustered]  treat:post = {m1.params['treat:post']:+.4f}, p = {m1.pvalues['treat:post']:.4f}")
print(f"[open, ticker-clustered] treat:post = {m2.params['treat:post']:+.4f}, p = {m2.pvalues['treat:post']:.4f}")

print()
print(
    panel_open.group_by(["participant_group", "is_near_event"])
    .agg(pl.col("share").mean().alias("mean_open_share"), pl.len().alias("n"))
    .sort("participant_group", "is_near_event")
)

Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
[open, event-clustered]  treat:post = +0.0322, p = 0.0000
[open, ticker-clustered] treat:post = +0.0322, p = 0.0000

shape: (4, 4)
┌───────────────────┬───────────────┬─────────────────┬───────┐
│ participant_group ┆ is_near_event ┆ mean_open_share ┆ n     │
│ ---               ┆ ---           ┆ ---             ┆ ---   │
│ str               ┆ bool          ┆ f64             ┆ u32   │
╞═══════════════════╪═══════════════╪═════════════════╪═══════╡
│ procust           ┆ false         ┆ 0.771756        ┆ 74481 │
│ procust           ┆ true          ┆ 0.740267        ┆ 48325 │
│ retail            ┆ false         ┆ 0.651122        ┆ 97197 │
│ retail            ┆ true          ┆ 0.651809        ┆ 97187 │
└───────────────────┴───────────────┴─────────────────┴───────┘


## Mega-cap divergence

Ten tickers (AAPL, AMD, AMZN, BAC, C, FB, GE, MSFT, NFLX, TSLA) behave
oppositely to the rest of the universe on position size, and carry enough
volume to reverse the sign of any volume-weighted aggregate. This is the
empirical justification for firm-event equal weighting and for the firm-size
control used throughout.

In [9]:
from analysis.event_window_profile import compare_weighting_schemes, compare_top_n_tickers

compare_weighting_schemes(outcome="lt_100")
print()
compare_top_n_tickers(outcome="lt_100", top_n=10)

Matched 97,270 of 163,010 events to CBOE trading data

=== Volume-weighted (every event-day counted by its own volume) ===
  retail: shift = +0.0142
  procust: shift = +0.0268

=== Equal-weighted per event (every firm-event counted once) ===
  retail: shift = +0.0758
  procust: shift = +0.0333

Top 10 tickers by volume: ['AAPL', 'AMD', 'AMZN', 'BAC', 'C', 'FB', 'GE', 'MSFT', 'NFLX', 'TSLA']

--- TOP 10 ---
  retail: shift = +0.0024
  procust: shift = +0.0529

--- EVERYONE ELSE ---
  retail: shift = +0.0761
  procust: shift = +0.0339



## Continuous dispersion regression

Replaces the binary near-event indicator with the continuous dispersion
measure. The `dispersion:treat:post` coefficient is the research question
stated as a number: does retail's near-event shift grow with analyst
disagreement, relative to professionals?

In [10]:
from analysis.event_window_profile import run_dispersion_regression

panel_otm = build_diff_in_diff_panel(outcome="otm")

print("=== Triple interaction ===")
m = run_dispersion_regression(panel_otm, spec="triple", cluster_by="ticker")
print(m.summary().tables[1])

print("\n=== Within earnings windows only ===")
m2 = run_dispersion_regression(panel_otm, spec="near_only", cluster_by="ticker")
print(m2.summary().tables[1])

Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
=== Triple interaction ===
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.5414      0.002    258.380      0.000       0.537       0.545
dispersion                0.0073      0.002      4.240      0.000       0.004       0.011
treat                     0.0526      0.002     27.966      0.000       0.049       0.056
dispersion:treat          0.0015      0.002      0.940      0.347      -0.002       0.005
post                      0.0021      0.002      0.992      0.321      -0.002       0.006
dispersion:post           0.0012      0.002      0.561      0.575      -0.003       0.005
treat:post                0.0104      0.002      4.737      0.000       0.006       0.015
dispersion:treat:post    -0.0046      0.002    

### RQ1 — does dispersion predict retail trading volume?

In [11]:
print("=== Log average daily volume ===")
mv = run_dispersion_regression(panel_otm, spec="triple", outcome_var="log_volume", cluster_by="ticker")
print(mv.summary().tables[1])

=== Log average daily volume ===
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.9573      0.033     29.397      0.000       0.893       1.021
dispersion               -0.0895      0.017     -5.317      0.000      -0.122      -0.056
treat                     3.6819      0.013    279.642      0.000       3.656       3.708
dispersion:treat          0.1250      0.010     12.915      0.000       0.106       0.144
post                      1.1803      0.016     74.140      0.000       1.149       1.212
dispersion:post           0.0320      0.012      2.586      0.010       0.008       0.056
treat:post               -0.9573      0.018    -53.076      0.000      -0.993      -0.922
dispersion:treat:post    -0.0918      0.013     -6.956      0.000      -0.118      -0.066


## Is the dispersion effect actually firm size?

Analyst dispersion is systematically lower for large firms, so an apparent
dispersion effect may be size in disguise. Size enters as a full triple
interaction rather than a level control, so that the dispersion coefficient
is identified holding size-dependent event response constant.

In [12]:
from analysis.event_window_profile import add_market_cap, exclude_top_n_tickers

print("=== 1. Baseline (no size control) ===")
m1 = run_dispersion_regression(panel_otm, spec="triple", cluster_by="ticker")
print(f"dispersion:treat:post = {m1.params['dispersion:treat:post']:+.4f}, p = {m1.pvalues['dispersion:treat:post']:.4f}")

print("\n=== 2. Excluding the ten divergent mega-caps ===")
panel_ex = exclude_top_n_tickers(panel_otm, n=10)
m2 = run_dispersion_regression(panel_ex, spec="triple", cluster_by="ticker")
print(f"dispersion:treat:post = {m2.params['dispersion:treat:post']:+.4f}, p = {m2.pvalues['dispersion:treat:post']:.4f}")

print("\n=== 3. Controlling for market cap ===")
panel_mc = add_market_cap(panel_otm)
m3 = run_dispersion_regression(panel_mc, spec="triple", cluster_by="ticker", controls=["log_mktcap"])
print(m3.summary().tables[1])

=== 1. Baseline (no size control) ===
dispersion:treat:post = -0.0046, p = 0.0379

=== 2. Excluding the ten divergent mega-caps ===
Excluding top 10 tickers by volume: ['AAPL', 'AMD', 'AMZN', 'BAC', 'C', 'FB', 'GE', 'MSFT', 'NFLX', 'TSLA']
  317,190 -> 315,465 panel rows
dispersion:treat:post = -0.0046, p = 0.0395

=== 3. Controlling for market cap ===
Market cap matched for 316,171 of 317,190 panel rows (99.7%)
  Dropped 1,019 rows missing control values (316,171 remain)
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.5413      0.002    258.693      0.000       0.537       0.545
dispersion                0.0136      0.002      7.974      0.000       0.010       0.017
treat                     0.0529      0.002     28.330      0.000       0.049       0.057
dispersion:treat         -0.0070      0.002     -4.393      0.000      -0

In [13]:
print("=== RQ1 volume, with size control ===")
mv = run_dispersion_regression(panel_mc, spec="triple", outcome_var="log_volume",
                               cluster_by="ticker", controls=["log_mktcap"])
print(mv.summary().tables[1])

=== RQ1 volume, with size control ===
  Dropped 1,019 rows missing control values (316,171 remain)
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.8913      0.023     38.204      0.000       0.846       0.937
dispersion                0.1170      0.014      8.454      0.000       0.090       0.144
treat                     3.8251      0.012    324.609      0.000       3.802       3.848
dispersion:treat          0.0729      0.009      7.728      0.000       0.054       0.091
post                      1.0185      0.013     75.854      0.000       0.992       1.045
dispersion:post          -0.0106      0.012     -0.852      0.394      -0.035       0.014
treat:post               -0.7812      0.015    -51.438      0.000      -0.811      -0.751
dispersion:treat:post    -0.0231      0.013     -1.787      0.074      -0.048       0.002
l

### Robustness across outcomes

`did_ctrl` collapsing while `size_did` is significant means firm size was
doing the work. `cross_ctrl` is the cross-sectional relationship, which is a
separate question from the event response and may survive when it does not.

In [14]:
from analysis.event_window_profile import dispersion_robustness_table

print("=== Composition outcomes (share) ===")
tbl = dispersion_robustness_table(outcomes=["otm", "otm_put", "lt_100", "call", "open"])
with pl.Config(tbl_cols=12, tbl_width_chars=220, float_precision=4):
    print(tbl)

=== Composition outcomes (share) ===
shape: (5, 9)
┌─────────┬──────────┬────────┬──────────┬────────┬────────────┬─────────┬──────────┬────────┐
│ outcome ┆ did_base ┆ p_base ┆ did_ctrl ┆ p_ctrl ┆ cross_ctrl ┆ p_cross ┆ size_did ┆ p_size │
│ ---     ┆ ---      ┆ ---    ┆ ---      ┆ ---    ┆ ---        ┆ ---     ┆ ---      ┆ ---    │
│ str     ┆ f64      ┆ f64    ┆ f64      ┆ f64    ┆ f64        ┆ f64     ┆ f64      ┆ f64    │
╞═════════╪══════════╪════════╪══════════╪════════╪════════════╪═════════╪══════════╪════════╡
│ otm     ┆ -0.0046  ┆ 0.0379 ┆ -0.0024  ┆ 0.2886 ┆ -0.0070    ┆ 0.0000  ┆ 0.0120   ┆ 0.0000 │
│ otm_put ┆ -0.0045  ┆ 0.0146 ┆ -0.0025  ┆ 0.1781 ┆ -0.0014    ┆ 0.2813  ┆ 0.0116   ┆ 0.0000 │
│ lt_100  ┆ 0.0027   ┆ 0.0710 ┆ 0.0004   ┆ 0.8095 ┆ -0.0031    ┆ 0.0238  ┆ -0.0116  ┆ 0.0000 │
│ call    ┆ 0.0023   ┆ 0.2997 ┆ -0.0001  ┆ 0.9509 ┆ -0.0013    ┆ 0.3854  ┆ -0.0146  ┆ 0.0000 │
│ open    ┆ -0.0055  ┆ 0.0054 ┆ -0.0047  ┆ 0.0197 ┆ 0.0027     ┆ 0.0628  ┆ 0.0023   ┆ 0.2261 │

In [15]:
print("=== Volume outcome (RQ1) ===")
tbl_vol = dispersion_robustness_table(outcomes=["otm"], outcome_var="log_volume")
with pl.Config(tbl_cols=12, tbl_width_chars=220, float_precision=4):
    print(tbl_vol)

=== Volume outcome (RQ1) ===
shape: (1, 9)
┌─────────┬──────────┬────────┬──────────┬────────┬────────────┬─────────┬──────────┬────────┐
│ outcome ┆ did_base ┆ p_base ┆ did_ctrl ┆ p_ctrl ┆ cross_ctrl ┆ p_cross ┆ size_did ┆ p_size │
│ ---     ┆ ---      ┆ ---    ┆ ---      ┆ ---    ┆ ---        ┆ ---     ┆ ---      ┆ ---    │
│ str     ┆ f64      ┆ f64    ┆ f64      ┆ f64    ┆ f64        ┆ f64     ┆ f64      ┆ f64    │
╞═════════╪══════════╪════════╪══════════╪════════╪════════════╪═════════╪══════════╪════════╡
│ otm     ┆ -0.0918  ┆ 0.0000 ┆ -0.0231  ┆ 0.0740 ┆ 0.0729     ┆ 0.0000  ┆ 0.3879   ┆ 0.0000 │
└─────────┴──────────┴────────┴──────────┴────────┴────────────┴─────────┴──────────┴────────┘


## Functional form

Dispersion is heavily right-skewed, so a linear coefficient may average a
null across most of the distribution with an effect confined to one tail.
The quartile specification lets the data choose its own shape.

In [16]:
panel_mc = add_market_cap(build_diff_in_diff_panel(outcome="otm"))
m = run_dispersion_regression(panel_mc, spec="near_only", cluster_by="ticker", controls=["log_mktcap"])
print(m.summary().tables[1])

Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
Market cap matched for 316,171 of 317,190 panel rows (99.7%)
  Dropped 460 rows missing control values (145,052 remain)
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.5382      0.002    224.193      0.000       0.533       0.543
dispersion           0.0145      0.002      6.899      0.000       0.010       0.019
treat                0.0693      0.002     31.895      0.000       0.065       0.074
dispersion:treat    -0.0094      0.002     -4.542      0.000      -0.014      -0.005
log_mktcap           0.0268      0.002     11.258      0.000       0.022       0.032
log_mktcap:treat    -0.0347      0.002    -15.921      0.000      -0.039      -0.030


### Raw quartile means

Unconditional on size, so this and the size-controlled regression answer
different questions and need not agree.

In [17]:
panel_q = panel_mc.with_columns(
    pl.col("dispersion_scaled").qcut(4, labels=["Q1_low","Q2","Q3","Q4_high"]).alias("disp_q")
).filter(pl.col("is_near_event"))

print(
    panel_q.group_by(["disp_q", "participant_group"])
    .agg(pl.col("share").mean().alias("mean_otm_share"), pl.len().alias("n"))
    .sort("disp_q", "participant_group")
)

shape: (8, 4)
┌─────────┬───────────────────┬────────────────┬───────┐
│ disp_q  ┆ participant_group ┆ mean_otm_share ┆ n     │
│ ---     ┆ ---               ┆ ---            ┆ ---   │
│ cat     ┆ str               ┆ f64            ┆ u32   │
╞═════════╪═══════════════════╪════════════════╪═══════╡
│ Q1_low  ┆ procust           ┆ 0.543361       ┆ 13813 │
│ Q1_low  ┆ retail            ┆ 0.596625       ┆ 23515 │
│ Q2      ┆ procust           ┆ 0.533005       ┆ 12655 │
│ Q2      ┆ retail            ┆ 0.597192       ┆ 23926 │
│ Q3      ┆ procust           ┆ 0.536354       ┆ 11297 │
│ Q3      ┆ retail            ┆ 0.61118        ┆ 24537 │
│ Q4_high ┆ procust           ┆ 0.561439       ┆ 10560 │
│ Q4_high ┆ retail            ┆ 0.620367       ┆ 25209 │
└─────────┴───────────────────┴────────────────┴───────┘


### How tightly are dispersion and size bound?

If mean market cap falls monotonically across dispersion quartiles, the two
variables are close to collinear and the independent dispersion effect is
identified off limited variation.

In [18]:
panel_q = panel_mc.with_columns(
    pl.col("dispersion_scaled").qcut(4, labels=["Q1_low","Q2","Q3","Q4_high"]).alias("disp_q")
)

print(
    panel_q.filter(pl.col("is_near_event"))
    .group_by("disp_q")
    .agg(
        pl.col("log_mktcap").mean().alias("mean_log_mktcap"),
        pl.col("dispersion_scaled").mean().alias("mean_dispersion"),
        pl.len().alias("n"),
    )
    .sort("disp_q")
)

shape: (4, 4)
┌─────────┬─────────────────┬─────────────────┬───────┐
│ disp_q  ┆ mean_log_mktcap ┆ mean_dispersion ┆ n     │
│ ---     ┆ ---             ┆ ---             ┆ ---   │
│ cat     ┆ f64             ┆ f64             ┆ u32   │
╞═════════╪═════════════════╪═════════════════╪═══════╡
│ Q1_low  ┆ 15.94004        ┆ 0.019422        ┆ 37328 │
│ Q2      ┆ 15.371297       ┆ 0.057118        ┆ 36581 │
│ Q3      ┆ 14.690766       ┆ 0.136579        ┆ 35834 │
│ Q4_high ┆ 14.338974       ┆ 0.62787         ┆ 35769 │
└─────────┴─────────────────┴─────────────────┴───────┘


### Log transformation and quartile dummies

Comparing the two shows how much the result depends on functional form. The
quartile dummies are assumption-free and are the specification reported.

In [19]:
panel_log = panel_mc.with_columns(
    (pl.col("dispersion_scaled") + 0.01).log().alias("dispersion_scaled")
)
m_log = run_dispersion_regression(
    panel_log, spec="near_only", cluster_by="ticker", controls=["log_mktcap"]
)
print("=== log(dispersion) ===")
print(m_log.summary().tables[1])

  Dropped 460 rows missing control values (145,052 remain)
=== log(dispersion) ===
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.5381      0.002    224.194      0.000       0.533       0.543
dispersion           0.0155      0.002      7.116      0.000       0.011       0.020
treat                0.0694      0.002     31.993      0.000       0.065       0.074
dispersion:treat    -0.0082      0.002     -3.944      0.000      -0.012      -0.004
log_mktcap           0.0291      0.002     11.890      0.000       0.024       0.034
log_mktcap:treat    -0.0356      0.002    -15.935      0.000      -0.040      -0.031


In [20]:
df = (
    panel_mc.filter(pl.col("is_near_event"))
    .with_columns(pl.col("dispersion_scaled").qcut(4, labels=["Q1","Q2","Q3","Q4"]).alias("disp_q"))
    .to_pandas()
    .dropna(subset=["log_mktcap"])
)
df["treat"] = (df["participant_group"] == "retail").astype(int)
df["log_mktcap_z"] = (df["log_mktcap"] - df["log_mktcap"].mean()) / df["log_mktcap"].std()

m_dum = smf.ols("share ~ C(disp_q) * treat + log_mktcap_z * treat", data=df).fit(
    cov_type="cluster", cov_kwds={"groups": df["resolved_ticker"]}
)
print(m_dum.summary().tables[1])

                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.5236      0.004    127.207      0.000       0.515       0.532
C(disp_q)[T.Q2]          -0.0002      0.005     -0.048      0.961      -0.010       0.010
C(disp_q)[T.Q3]           0.0134      0.006      2.408      0.016       0.002       0.024
C(disp_q)[T.Q4]           0.0449      0.006      7.630      0.000       0.033       0.056
treat                     0.0762      0.004     19.632      0.000       0.069       0.084
C(disp_q)[T.Q2]:treat    -0.0003      0.005     -0.060      0.952      -0.010       0.009
C(disp_q)[T.Q3]:treat    -0.0024      0.005     -0.454      0.650      -0.013       0.008
C(disp_q)[T.Q4]:treat    -0.0246      0.006     -4.409      0.000      -0.035      -0.014
log_mktcap_z              0.0296      0.002     12.070      0.000       0.025       0.034
log_mktcap

### Quartile specification on volume

Volume is outcome-independent here (`avg_daily_vol = total_vol / n_days`),
so the panel built for `otm` serves. Both periods are reported: if they look
alike, the dispersion–volume relationship is structural rather than
event-driven.

In [21]:
df_all = (
    panel_mc
    .filter(pl.col("avg_daily_vol") > 0)
    .with_columns(pl.col("dispersion_scaled").qcut(4, labels=["Q1","Q2","Q3","Q4"]).alias("disp_q"))
    .to_pandas()
    .dropna(subset=["log_mktcap"])
)
df_all["treat"] = (df_all["participant_group"] == "retail").astype(int)
df_all["log_mktcap_z"] = (df_all["log_mktcap"] - df_all["log_mktcap"].mean()) / df_all["log_mktcap"].std()
df_all["y"] = np.log(df_all["avg_daily_vol"])

for label, sub in [("NEAR-EVENT", df_all[df_all["is_near_event"]]),
                   ("BASELINE",   df_all[~df_all["is_near_event"]])]:
    m = smf.ols("y ~ C(disp_q) * treat + log_mktcap_z * treat", data=sub).fit(
        cov_type="cluster", cov_kwds={"groups": sub["resolved_ticker"]}
    )
    print(f"=== {label} (n = {int(m.nobs):,}) ===")
    print(m.summary().tables[1])
    print()

=== NEAR-EVENT (n = 145,052) ===
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 1.7038      0.042     40.177      0.000       1.621       1.787
C(disp_q)[T.Q2]           0.0744      0.047      1.577      0.115      -0.018       0.167
C(disp_q)[T.Q3]           0.3029      0.054      5.596      0.000       0.197       0.409
C(disp_q)[T.Q4]           0.4410      0.061      7.266      0.000       0.322       0.560
treat                     2.9204      0.030     97.461      0.000       2.862       2.979
C(disp_q)[T.Q2]:treat     0.0701      0.032      2.177      0.029       0.007       0.133
C(disp_q)[T.Q3]:treat     0.1738      0.037      4.723      0.000       0.102       0.246
C(disp_q)[T.Q4]:treat     0.2664      0.042      6.273      0.000       0.183       0.350
log_mktcap_z              0.8116      0.034     24.067      0.000  

## The near-event firm-size reversal

Retail's volume advantage over professionals shrinks with firm size in
ordinary conditions but grows with it around earnings. Three checks follow:
whether this is sample selection, whether it is a gradient or a threshold,
and which group actually moves.

In [22]:
from analysis.event_window_profile import investigate_size_reversal

panel_mc = add_market_cap(build_diff_in_diff_panel(outcome="otm"))
res = investigate_size_reversal(panel_mc)

for name, tbl in res.items():
    print(f"=== {name} ===")
    with pl.Config(tbl_rows=20, float_precision=3):
        print(tbl)
    print()

Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
Market cap matched for 316,171 of 317,190 panel rows (99.7%)
=== selection ===
shape: (3, 3)
┌───────────────┬─────────────────┬────────┐
│ presence      ┆ mean_log_mktcap ┆ n      │
│ ---           ┆ ---             ┆ ---    │
│ str           ┆ f64             ┆ u32    │
╞═══════════════╪═════════════════╪════════╡
│ baseline_only ┆ 14.362          ┆ 28033  │
│ both          ┆ 15.110          ┆ 143086 │
│ near_only     ┆ 14.122          ┆ 1966   │
└───────────────┴─────────────────┴────────┘

=== by_size_quartile ===
shape: (16, 5)
┌──────────┬───────────────────┬───────────────┬──────────────┬───────┐
│ size_q   ┆ participant_group ┆ is_near_event ┆ mean_log_vol ┆ n     │
│ ---      ┆ ---               ┆ ---           ┆ ---          ┆ ---   │
│ cat      ┆ str               ┆ bool          ┆ f64          ┆ u32   │
╞══════════╪═══════════════════╪═══════════════╪═════════

### Balanced panel

A panel row exists only where that group traded in that period. Professional
customers drop out of the six-day near-event window far more often than the
~55-day baseline, and at a rate falling with firm size — so estimates are
partly conditioned on "professionals happened to trade". The balanced panel
removes that conditioning.

In [23]:
complete = (
    panel_mc.filter(pl.col("log_mktcap").is_not_null() & (pl.col("avg_daily_vol") > 0))
    .group_by(["resolved_ticker", "ANNDATS_ACT"])
    .agg(pl.len().alias("n_cells"))
    .filter(pl.col("n_cells") == 4)
    .select(["resolved_ticker", "ANNDATS_ACT"])
)
balanced = panel_mc.join(complete, on=["resolved_ticker", "ANNDATS_ACT"], how="inner")
print(f"Balanced panel: {balanced.height:,} rows (from {panel_mc.height:,})")

df = balanced.filter(pl.col("avg_daily_vol") > 0).to_pandas().dropna(subset=["log_mktcap"])
df["treat"] = (df["participant_group"] == "retail").astype(int)
df["post"] = df["is_near_event"].astype(int)
df["log_mktcap_z"] = (df["log_mktcap"] - df["log_mktcap"].mean()) / df["log_mktcap"].std()
df["y"] = np.log(df["avg_daily_vol"])

m = smf.ols("y ~ log_mktcap_z * treat * post", data=df).fit(
    cov_type="cluster", cov_kwds={"groups": df["resolved_ticker"]}
)
print(m.summary().tables[1])

Balanced panel: 184,868 rows (from 317,190)
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                   1.8286      0.027     66.766      0.000       1.775       1.882
log_mktcap_z                0.8933      0.030     29.783      0.000       0.835       0.952
treat                       3.7081      0.011    332.203      0.000       3.686       3.730
log_mktcap_z:treat         -0.1182      0.011    -10.995      0.000      -0.139      -0.097
post                        0.3761      0.010     36.287      0.000       0.356       0.396
log_mktcap_z:post          -0.1787      0.010    -17.418      0.000      -0.199      -0.159
treat:post                  0.1115      0.010     10.833      0.000       0.091       0.132
log_mktcap_z:treat:post     0.2024      0.010     20.259      0.000       0.183       0.222


### Headline results on the balanced panel

All outcomes hold under both specifications with the same signs. Note
`lt_100` changes substantially, which is why its instability is reported
rather than a single value being chosen.

In [24]:
for oc in ["otm", "otm_put", "lt_100", "call", "open"]:
    p_full = build_diff_in_diff_panel(outcome=oc, verbose=False)
    p_bal = p_full.join(complete, on=["resolved_ticker", "ANNDATS_ACT"], how="inner")
    m_f = run_diff_in_diff(p_full, cluster_by="ticker")
    m_b = run_diff_in_diff(p_bal, cluster_by="ticker")
    print(f"{oc:9s} full = {m_f.params['treat:post']:+.4f} (p={m_f.pvalues['treat:post']:.4f})  "
          f"balanced = {m_b.params['treat:post']:+.4f} (p={m_b.pvalues['treat:post']:.4f})")

otm       full = +0.0107 (p=0.0000)  balanced = +0.0209 (p=0.0000)
otm_put   full = +0.0097 (p=0.0000)  balanced = +0.0182 (p=0.0000)
lt_100    full = +0.0425 (p=0.0000)  balanced = +0.0105 (p=0.0000)
call      full = -0.0046 (p=0.0321)  balanced = -0.0148 (p=0.0000)
open      full = +0.0322 (p=0.0000)  balanced = +0.0230 (p=0.0000)


In [3]:
import polars as pl
from paths import DATA_DIR, CRSP_DIR, IBES_DIR

# 1. Raw CBOE: exact buy/sell column names for the two customer groups
cboe = pl.scan_parquet(DATA_DIR / "cboe_parquet" / "cboe_openclose_2021.parquet").collect_schema().names()
print("non-volume columns:", [c for c in cboe if not c.endswith("_vol")])
print("cust_:   ", [c for c in cboe if c.startswith("cust_")])
print("procust_:", [c for c in cboe if c.startswith("procust_")])

# 2. CRSP: confirm there are no return columns
crsp = pl.scan_parquet(CRSP_DIR / "crsp_daily.parquet").collect_schema().names()
print("CRSP return columns:", [c for c in crsp if "ret" in c.lower()])

# 3. Events: is announcement time carried through, and how are firms identified?
ev = pl.scan_parquet(IBES_DIR / "dispersion_events.parquet").collect_schema().names()
print("event columns:", ev)

non-volume columns: ['quote_date', 'underlying_symbol', 'security_type', 'option_symbol', 'expiration_date', 'strike_price', 'call_put_flag', 'days_to_expire', 'series_type', 'first_trade_price', 'high_trade_price', 'low_trade_price', 'last_trade_price', 'open_interest', 'previous_open_interest', 'previous_close', 'firm_open_buy_qty', 'firm_close_buy_qty', 'firm_open_sell_qty', 'firm_close_sell_qty', 'bd_open_buy_qty', 'bd_close_buy_qty', 'bd_open_sell_qty', 'bd_close_sell_qty', 'mm_buy_qty', 'mm_sell_qty', 'cust_lt_100_open_buy_qty', 'cust_lt_100_close_buy_qty', 'cust_lt_100_open_sell_qty', 'cust_lt_100_close_sell_qty', 'cust_100_199_open_buy_qty', 'cust_100_199_close_buy_qty', 'cust_100_199_open_sell_qty', 'cust_100_199_close_sell_qty', 'cust_gt_199_open_buy_qty', 'cust_gt_199_close_buy_qty', 'cust_gt_199_open_sell_qty', 'cust_gt_199_close_sell_qty', 'procust_lt_100_open_buy_qty', 'procust_lt_100_close_buy_qty', 'procust_lt_100_open_sell_qty', 'procust_lt_100_close_sell_qty', 'procus

In [2]:
import wrds
db = wrds.Connection()   # prompts for WRDS username, password and Duo on first run
print(db.raw_sql("SELECT MIN(date), MAX(date), COUNT(*) FROM optionm_all.opprcd2025"))

Enter your WRDS username [axels]: asparn
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


pgpass file created at C:\Users\axels\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
          min         max      count
0  2025-01-02  2025-08-29  264945680


In [3]:
import wrds
from pipeline.pull_optionmetrics import run_optionmetrics_pull

db = wrds.Connection()   # WRDS login + Duo the first time
run_optionmetrics_pull(db, years=[2021], max_chunks=1)

Enter your WRDS username [axels]: asparn
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


pgpass file created at C:\Users\axels\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Pulling secnmd (security names) ...
  272,255 name records, 120,578 secids
Events 2016-01-01 .. 2025-08-27: 142,697
  linked to an OptionMetrics secid: 135,363 (94.9%); 2 CUSIPs map to >1 secid over time (matched as of event date)
  541,452 (event, day) rows -> 535,336 unique (secid, date) pairs
Approximately 347797472 rows in optionm_all.opprcd2021.
Approximately 2325672 rows in optionm_all.secprd2021.
[2021] 57,876 pairs in 1 chunk(s) (tables: opprcd2021, secprd2021)
    chunk 1/1: 536,246 option rows so far (43s)

Done. Option prices in data/optionmetrics/opprcd_<year>/


In [4]:
import polars as pl
from paths import DATA_DIR

om = DATA_DIR / "optionmetrics"
f = om / "opprcd_2021" / "part_0000.parquet"
opt = pl.read_parquet(f)

print(opt.columns)
print(opt.select(
    pl.len().alias("rows"),
    pl.col("date").min().alias("first_date"),
    pl.col("date").max().alias("last_date"),
    (pl.col("exdate") - pl.col("date")).dt.total_days().max().alias("max_days_to_expiry"),
    pl.col("strike_price").median().alias("median_strike_x1000"),
    pl.col("delta").min().alias("delta_min"),
    pl.col("delta").max().alias("delta_max"),
    pl.col("impl_volatility").null_count().alias("iv_nulls"),
    (pl.col("impl_volatility") < 0).sum().alias("iv_negative"),
))
print(f"file size: {f.stat().st_size / 1e6:.1f} MB")

['secid', 'date', 'optionid', 'symbol', 'root', 'exdate', 'cp_flag', 'strike_price', 'best_bid', 'best_offer', 'volume', 'open_interest', 'impl_volatility', 'delta', 'gamma', 'vega', 'theta', 'contract_size', 'ss_flag', 'am_settlement', 'expiry_indicator']
shape: (1, 9)
┌────────┬────────────┬────────────┬────────────┬───┬───────────┬───────────┬──────────┬───────────┐
│ rows   ┆ first_date ┆ last_date  ┆ max_days_t ┆ … ┆ delta_min ┆ delta_max ┆ iv_nulls ┆ iv_negati │
│ ---    ┆ ---        ┆ ---        ┆ o_expiry   ┆   ┆ ---       ┆ ---       ┆ ---      ┆ ve        │
│ u32    ┆ datetime[n ┆ datetime[n ┆ ---        ┆   ┆ f64       ┆ f64       ┆ u32      ┆ ---       │
│        ┆ s]         ┆ s]         ┆ i64        ┆   ┆           ┆           ┆          ┆ u32       │
╞════════╪════════════╪════════════╪════════════╪═══╪═══════════╪═══════════╪══════════╪═══════════╡
│ 536246 ┆ 2021-01-08 ┆ 2021-12-23 ┆ 120        ┆ … ┆ -0.999986 ┆ 0.999998  ┆ 110500   ┆ 0         │
│        ┆ 00:00:00   

In [5]:
run_optionmetrics_pull(db)

Events 2016-01-01 .. 2025-08-27: 142,697
  linked to an OptionMetrics secid: 135,363 (94.9%); 2 CUSIPs map to >1 secid over time (matched as of event date)
  541,452 (event, day) rows -> 535,336 unique (secid, date) pairs
Approximately 215270192 rows in optionm_all.opprcd2016.
Approximately 2133974 rows in optionm_all.secprd2016.
[2016] 54,756 pairs in 28 chunk(s) (tables: opprcd2016, secprd2016)
    chunk 10/28: 2,970,993 option rows so far (258s)
    chunk 20/28: 5,358,332 option rows so far (456s)
    chunk 28/28: 6,476,905 option rows so far (557s)
Approximately 211592048 rows in optionm_all.opprcd2017.
Approximately 2097941 rows in optionm_all.secprd2017.
[2017] 53,165 pairs in 27 chunk(s) (tables: opprcd2017, secprd2017)
    chunk 10/27: 3,104,295 option rows so far (259s)
    chunk 20/27: 5,584,243 option rows so far (471s)
    chunk 27/27: 6,407,183 option rows so far (549s)
Approximately 225834624 rows in optionm_all.opprcd2018.
Approximately 2030503 rows in optionm_all.secprd

In [3]:
import polars as pl
from paths import DATA_DIR

om = DATA_DIR / "optionmetrics"
expected = {2016: 28, 2017: 27, 2018: 27, 2019: 27, 2020: 27,
            2021: 29, 2022: 28, 2023: 29, 2024: 28, 2025: 21}

# 1. every chunk saved, for both tables
for y, n_exp in expected.items():
    n_opt = len(list((om / f"opprcd_{y}").glob("part_*.parquet")))
    n_sec = len(list((om / f"secprd_{y}").glob("part_*.parquet")))
    rows = pl.scan_parquet(om / f"opprcd_{y}" / "*.parquet").select(pl.len()).collect().item()
    flag = "" if n_opt == n_sec == n_exp else "   <-- MISSING CHUNKS"
    print(f"{y}: {n_opt}/{n_exp} option parts, {n_sec}/{n_exp} security parts, {rows:,} rows{flag}")

# 2. conventions: IV as a decimal, and the pre-July-2016 vega placeholder
opt16 = pl.scan_parquet(om / "opprcd_2016" / "*.parquet")
print(opt16.select(
    pl.col("impl_volatility").median().alias("iv_median"),
    (pl.col("vega") < 0).sum().alias("vega_negative"),
    ((pl.col("vega") < 0) & (pl.col("date") < pl.datetime(2016, 7, 5))).sum().alias("vega_neg_pre_jul16"),
    pl.col("ss_flag").value_counts().alias("ss_flag_counts"),
).collect())

2016: 28/28 option parts, 28/28 security parts, 6,476,905 rows
2017: 27/27 option parts, 27/27 security parts, 6,407,183 rows
2018: 27/27 option parts, 27/27 security parts, 6,781,895 rows
2019: 27/27 option parts, 27/27 security parts, 6,718,983 rows
2020: 27/27 option parts, 27/27 security parts, 7,993,970 rows
2021: 29/29 option parts, 29/29 security parts, 9,154,133 rows
2022: 28/28 option parts, 28/28 security parts, 7,646,040 rows
2023: 29/29 option parts, 29/29 security parts, 8,376,960 rows
2024: 28/28 option parts, 28/28 security parts, 8,970,123 rows
2025: 21/21 option parts, 21/21 security parts, 6,764,407 rows
shape: (2, 4)
┌───────────┬───────────────┬────────────────────┬────────────────┐
│ iv_median ┆ vega_negative ┆ vega_neg_pre_jul16 ┆ ss_flag_counts │
│ ---       ┆ ---           ┆ ---                ┆ ---            │
│ f64       ┆ u32           ┆ u32                ┆ struct[2]      │
╞═══════════╪═══════════════╪════════════════════╪════════════════╡
│ 0.531963  ┆ 0 

In [4]:
import polars as pl
from paths import DATA_DIR, CRSP_DIR, IBES_DIR

# 1. Raw CBOE: exact buy/sell column names for the two customer groups
cboe = pl.scan_parquet(DATA_DIR / "cboe_parquet" / "cboe_openclose_2021.parquet").collect_schema().names()
print("non-volume columns:", [c for c in cboe if not c.endswith("_vol")])
print("cust_:   ", [c for c in cboe if c.startswith("cust_")])
print("procust_:", [c for c in cboe if c.startswith("procust_")])

# 2. CRSP: confirm there are no return columns
crsp = pl.scan_parquet(CRSP_DIR / "crsp_daily.parquet").collect_schema().names()
print("CRSP return columns:", [c for c in crsp if "ret" in c.lower()])

# 3. Events: is announcement time carried through, and how are firms identified?
ev = pl.scan_parquet(IBES_DIR / "dispersion_events.parquet").collect_schema().names()
print("event columns:", ev)

non-volume columns: ['quote_date', 'underlying_symbol', 'security_type', 'option_symbol', 'expiration_date', 'strike_price', 'call_put_flag', 'days_to_expire', 'series_type', 'first_trade_price', 'high_trade_price', 'low_trade_price', 'last_trade_price', 'open_interest', 'previous_open_interest', 'previous_close', 'firm_open_buy_qty', 'firm_close_buy_qty', 'firm_open_sell_qty', 'firm_close_sell_qty', 'bd_open_buy_qty', 'bd_close_buy_qty', 'bd_open_sell_qty', 'bd_close_sell_qty', 'mm_buy_qty', 'mm_sell_qty', 'cust_lt_100_open_buy_qty', 'cust_lt_100_close_buy_qty', 'cust_lt_100_open_sell_qty', 'cust_lt_100_close_sell_qty', 'cust_100_199_open_buy_qty', 'cust_100_199_close_buy_qty', 'cust_100_199_open_sell_qty', 'cust_100_199_close_sell_qty', 'cust_gt_199_open_buy_qty', 'cust_gt_199_close_buy_qty', 'cust_gt_199_open_sell_qty', 'cust_gt_199_close_sell_qty', 'procust_lt_100_open_buy_qty', 'procust_lt_100_close_buy_qty', 'procust_lt_100_open_sell_qty', 'procust_lt_100_close_sell_qty', 'procus

In [5]:
import wrds
import polars as pl
from paths import CRSP_DIR

db = wrds.Connection(wrds_username="asparn")   # the .pgpass file means no password prompt
out = CRSP_DIR / "returns"
out.mkdir(parents=True, exist_ok=True)

# check the table and its return columns before pulling
tabs = set(db.list_tables(library="crsp"))
assert "dsf_v2" in tabs, f"dsf_v2 not found; daily tables available: {sorted(t for t in tabs if t.startswith('dsf'))}"
cols = {c.lower() for c in db.describe_table(library="crsp", table="dsf_v2")["name"]}
print("return-related columns:", sorted(c for c in cols if "ret" in c))

for year in range(2010, 2026):
    f = out / f"crsp_returns_{year}.parquet"
    if f.exists():
        continue
    df = db.raw_sql(f"""
        SELECT permno, dlycaldt, dlyret, dlyretx
        FROM crsp.dsf_v2
        WHERE dlycaldt BETWEEN '{year}-01-01' AND '{year}-12-31'
    """, date_cols=["dlycaldt"])
    (pl.from_pandas(df)
       .rename({"permno": "PERMNO", "dlycaldt": "DlyCalDt", "dlyret": "DlyRet", "dlyretx": "DlyRetx"})
       .with_columns(pl.col("PERMNO").cast(pl.Int64), pl.col("DlyCalDt").cast(pl.Date))
       .write_parquet(f, compression="zstd"))
    print(f"{year}: {len(df):,} rows")

Loading library list...
Done
Approximately 110257376 rows in crsp.dsf_v2.
return-related columns: ['dlydistretflg', 'dlyret', 'dlyretdurflg', 'dlyreti', 'dlyretmissflg', 'dlyretx', 'sharetype']
2010: 1,687,536 rows
2011: 1,706,875 rows
2012: 1,701,795 rows
2013: 1,709,728 rows
2014: 1,771,529 rows
2015: 1,821,381 rows
2016: 1,829,837 rows
2017: 1,829,117 rows
2018: 1,870,828 rows
2019: 1,913,322 rows
2020: 1,950,357 rows
2021: 2,188,964 rows
2022: 2,391,799 rows
2023: 2,356,072 rows
2024: 2,403,637 rows
2025: 2,541,801 rows


In [6]:
dsi = db.raw_sql("""
    SELECT date, vwretd, vwretx, ewretd, sprtrn
    FROM crsp.dsi
    WHERE date >= '2010-01-01'
""", date_cols=["date"])
mkt = (pl.from_pandas(dsi)
         .rename({"date": "DlyCalDt"})
         .with_columns(pl.col("DlyCalDt").cast(pl.Date)))
mkt.write_parquet(out / "crsp_market_daily.parquet", compression="zstd")
print(f"market series: {mkt.height:,} days, {mkt['DlyCalDt'].min()} to {mkt['DlyCalDt'].max()}")

market series: 3,774 days, 2010-01-04 to 2024-12-31


In [7]:
import polars as pl
from pipeline.build_market_return import build_market_return, validate_market_return

mkt = build_market_return()
val = validate_market_return(mkt)
with pl.Config(tbl_rows=-1, float_precision=4):
    print(val)

Constructed market return: 4,023 days, 2010-01-05 to 2025-12-31
  median securities per day: all 7,300, common 4,425
  most common SecurityType/ShareType pairs in CRSP (sanity check on the filter):
shape: (6, 3)
┌──────────────┬───────────┬──────────┐
│ SecurityType ┆ ShareType ┆ len      │
│ ---          ┆ ---       ┆ ---      │
│ str          ┆ str       ┆ u32      │
╞══════════════╪═══════════╪══════════╡
│ EQTY         ┆ NS        ┆ 18849315 │
│ FUND         ┆ NS        ┆ 9541278  │
│ EQTY         ┆ AD        ┆ 1474933  │
│ FUND         ┆ SB        ┆ 1178664  │
│ EQTY         ┆ UG        ┆ 459363   │
│ EQTY         ┆ SB        ┆ 155047   │
└──────────────┴───────────┴──────────┘
mkt_vw_all      vs vwretd: corr 0.99981, mean |diff| 1.53 bp/day over 3,773 days
mkt_vw_common   vs vwretd: corr 0.99981, mean |diff| 2.56 bp/day over 3,773 days
shape: (30, 4)
┌──────┬────────┬──────────────────┬───────────────┐
│ year ┆ corr   ┆ mean_abs_diff_bp ┆ series        │
│ ---  ┆ ---    ┆ ---    

In [9]:
from pipeline.build_daily_flow import build_daily_flow
from paths import DATA_DIR

build_daily_flow()                                          # base   -> data/cboe_daily_flow/
build_daily_flow(cboe_dir=DATA_DIR / "cboe_parquet_ext",
                 out_dir=DATA_DIR / "cboe_daily_flow_ext")  # extension -> data/cboe_daily_flow_ext/

cboe_openclose_2011.parquet: -> 537,987 ticker-days -> daily_flow_2011.parquet (1.1s)
cboe_openclose_2012.parquet: -> 503,887 ticker-days -> daily_flow_2012.parquet (0.9s)
cboe_openclose_2013.parquet: -> 520,012 ticker-days -> daily_flow_2013.parquet (0.9s)
cboe_openclose_2014.parquet: -> 563,196 ticker-days -> daily_flow_2014.parquet (1.2s)
cboe_openclose_2015.parquet: -> 510,751 ticker-days -> daily_flow_2015.parquet (1.1s)
cboe_openclose_2016.parquet: -> 512,233 ticker-days -> daily_flow_2016.parquet (1.1s)
cboe_openclose_2017.parquet: -> 531,608 ticker-days -> daily_flow_2017.parquet (1.3s)
cboe_openclose_2018.parquet: -> 579,148 ticker-days -> daily_flow_2018.parquet (1.5s)
cboe_openclose_2019.parquet: -> 585,280 ticker-days -> daily_flow_2019.parquet (1.6s)
cboe_openclose_2020.parquet: -> 648,728 ticker-days -> daily_flow_2020.parquet (2.3s)
cboe_openclose_2021.parquet: -> 778,796 ticker-days -> daily_flow_2021.parquet (2.6s)
cboe_openclose_2022.parquet: -> 293,530 ticker-days ->

underlying_symbol,quote_date,retail_call_open_buy,retail_call_close_buy,retail_call_open_sell,retail_call_close_sell,retail_put_open_buy,retail_put_close_buy,retail_put_open_sell,retail_put_close_sell,procust_call_open_buy,procust_call_close_buy,procust_call_open_sell,procust_call_close_sell,procust_put_open_buy,procust_put_close_buy,procust_put_open_sell,procust_put_close_sell
str,date,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""A""",2022-08-01,44,7,2,7,1,6,5,6,0,0,0,0,0,0,0,0
"""A""",2022-08-02,47,5,0,6,0,0,0,0,0,0,0,0,10,0,10,0
"""A""",2022-08-03,0,8,0,23,0,0,0,0,0,0,0,0,0,10,6,10
"""A""",2022-08-04,1,8,0,9,0,0,0,2,0,0,0,0,0,0,9,0
"""A""",2022-08-05,1,3,2,3,3,14,0,13,0,0,6,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""^XSP""",2026-05-22,27916,7309,13918,17640,34344,40802,46623,24940,560,460,920,198,511,630,1048,194
"""^XSP""",2026-05-26,34438,8212,15989,21256,39342,14410,28231,19949,504,323,893,287,522,808,1594,324
"""^XSP""",2026-05-27,25195,4738,10927,14907,28043,9023,17922,17756,206,77,252,90,315,197,328,137


In [10]:
print("base files:", len(list((DATA_DIR / "cboe_daily_flow").glob("daily_flow_*.parquet"))))
print("ext files: ", len(list((DATA_DIR / "cboe_daily_flow_ext").glob("daily_flow_*.parquet"))))

base files: 12
ext files:  5


In [11]:
panel = build_event_panel()
tbl = informed_trading_table(panel)
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=4, tbl_width_chars=200):
    print(tbl)

Events 2016-01-01 .. 2025-12-31: 146,523
  dropped 104 whose flow window falls in a CBOE data gap
  with a complete flow window: 146,419  |  linked to a PERMNO: 140,906
  with a complete CAR[0,+1]: 140,684
  announcement timing: after_close 138,767, during_market 2,139
  events with retail option volume in [-2,-1]: 96,054
  events with procust option volume in [-2,-1]: 32,293
shape: (16, 11)
┌─────────────────┬─────────┬────────┬────────────┬───────┬──────────┬───────────┬──────────┬───────────┬────────────┬───────────┐
│ dv              ┆ flow    ┆ sample ┆ era        ┆ n     ┆ b_retail ┆ se_retail ┆ p_retail ┆ b_procust ┆ se_procust ┆ p_procust │
│ ---             ┆ ---     ┆ ---    ┆ ---        ┆ ---   ┆ ---      ┆ ---       ┆ ---      ┆ ---       ┆ ---        ┆ ---       │
│ str             ┆ str     ┆ str    ┆ str        ┆ i64   ┆ f64      ┆ f64       ┆ f64      ┆ f64       ┆ f64        ┆ f64       │
╞═════════════════╪═════════╪════════╪════════════╪═══════╪══════════╪═══════════

In [2]:
import polars as pl
from paths import IBES_DIR

t = pl.read_parquet(IBES_DIR / "dispersion_events.parquet", columns=["ANNTIMS_ACT"])
print(t.schema)
print(t.select(pl.col("ANNTIMS_ACT").cast(pl.Utf8).str.len_chars().alias("length"))
       .group_by("length").len().sort("length"))
print(t["ANNTIMS_ACT"].drop_nulls().sample(10, seed=1).to_list())

Schema({'ANNTIMS_ACT': String})
shape: (2, 2)
┌────────┬───────┐
│ length ┆ len   │
│ ---    ┆ ---   │
│ u32    ┆ u32   │
╞════════╪═══════╡
│ 7      ┆ 72159 │
│ 8      ┆ 90860 │
└────────┴───────┘
['16:05:00', '6:30:00', '6:00:00', '8:05:00', '6:43:00', '7:00:00', '16:06:00', '7:00:00', '17:00:00', '16:02:00']


In [3]:
import polars as pl
from analysis.informed_trading import build_event_panel, informed_trading_table

panel = build_event_panel()
tbl = informed_trading_table(panel)
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=4, tbl_width_chars=200):
    print(tbl)

Events 2016-01-01 .. 2025-12-31: 146,523
  dropped 194 whose flow window falls in a CBOE data gap
  with a complete flow window: 146,329  |  linked to a PERMNO: 140,817
  with a complete CAR[0,+1]: 140,601
  announcement timing: after_close 78,180, before_open 60,360, during_market 2,264, unknown 13
  events with retail option volume in [-2,-1]: 94,039
  events with procust option volume in [-2,-1]: 31,141
shape: (16, 11)
┌─────────────────┬─────────┬────────┬────────────┬───────┬──────────┬───────────┬──────────┬───────────┬────────────┬───────────┐
│ dv              ┆ flow    ┆ sample ┆ era        ┆ n     ┆ b_retail ┆ se_retail ┆ p_retail ┆ b_procust ┆ se_procust ┆ p_procust │
│ ---             ┆ ---     ┆ ---    ┆ ---        ┆ ---   ┆ ---      ┆ ---       ┆ ---      ┆ ---       ┆ ---        ┆ ---       │
│ str             ┆ str     ┆ str    ┆ str        ┆ i64   ┆ f64      ┆ f64       ┆ f64      ┆ f64       ┆ f64        ┆ f64       │
╞═════════════════╪═════════╪════════╪════════════

In [4]:
# 1. robustness to the flow definition (pp_ratio is the Pan & Poteshman opening-buy measure)
rob = informed_trading_table(panel, dvs=("car01",),
                             flows=("net_put", "pp_ratio", "net_bearish"), samples=("retail",))

# 2. firm-size terciles
sized = panel.with_columns(pl.col("log_mktcap").qcut(3, labels=["small", "mid", "large"]).alias("size_t"))
size_rows = []
for s in ["small", "mid", "large"]:
    t = informed_trading_table(sized.filter(pl.col("size_t") == s), dvs=("car01",), samples=("retail",))
    size_rows.append(t.with_columns(pl.lit(s).alias("size")))
size_tbl = pl.concat(size_rows)

# 3. events with vs without professional trading
pro_rows = []
for label, sub in [("with_pro", panel.filter(pl.col("procust_net_put").is_not_null())),
                   ("no_pro", panel.filter(pl.col("procust_net_put").is_null()))]:
    t = informed_trading_table(sub, dvs=("car01",), samples=("retail",))
    pro_rows.append(t.with_columns(pl.lit(label).alias("subset")))
pro_tbl = pl.concat(pro_rows)

with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=4, tbl_width_chars=200):
    for name, t in [("flow measures", rob), ("size terciles", size_tbl), ("professional presence", pro_tbl)]:
        print(f"\n=== {name} ===")
        print(t.select([c for c in ["flow", "size", "subset", "era", "n", "b_retail", "se_retail", "p_retail"] if c in t.columns]))


=== flow measures ===
shape: (12, 6)
┌─────────────┬────────────┬───────┬──────────┬───────────┬──────────┐
│ flow        ┆ era        ┆ n     ┆ b_retail ┆ se_retail ┆ p_retail │
│ ---         ┆ ---        ┆ ---   ┆ ---      ┆ ---       ┆ ---      │
│ str         ┆ str        ┆ i64   ┆ f64      ┆ f64       ┆ f64      │
╞═════════════╪════════════╪═══════╪══════════╪═══════════╪══════════╡
│ net_bearish ┆ pooled     ┆ 94000 ┆ -0.0007  ┆ 0.0003    ┆ 0.0420   │
│ net_bearish ┆ pre_covid  ┆ 34857 ┆ -0.0021  ┆ 0.0005    ┆ 0.0000   │
│ net_bearish ┆ covid      ┆ 19476 ┆ -0.0007  ┆ 0.0006    ┆ 0.2748   │
│ net_bearish ┆ post_covid ┆ 34504 ┆ 0.0007   ┆ 0.0007    ┆ 0.3132   │
│ net_put     ┆ pooled     ┆ 94000 ┆ -0.0007  ┆ 0.0003    ┆ 0.0233   │
│ net_put     ┆ pre_covid  ┆ 34857 ┆ -0.0016  ┆ 0.0004    ┆ 0.0003   │
│ net_put     ┆ covid      ┆ 19476 ┆ -0.0006  ┆ 0.0006    ┆ 0.3461   │
│ net_put     ┆ post_covid ┆ 34504 ┆ 0.0002   ┆ 0.0006    ┆ 0.7736   │
│ pp_ratio    ┆ pooled     ┆ 84206 ┆ -0

In [5]:
import polars as pl
from analysis.informed_trading import build_event_panel
from analysis.option_returns import build_option_panel, event_option_returns, option_returns_table

panel = build_event_panel(verbose=False)       # skip if `panel` is still in memory
contracts = build_option_panel(panel)          # contract-level panel, also saved for Test 3
er = event_option_returns(contracts)
tbl = option_returns_table(er)

with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=4, tbl_width_chars=250):
    print(tbl.select("measure", "era", "retail_n", "retail_mean", "retail_p", "retail_median",
                     "procust_n", "procust_mean", "procust_median", "diff_n", "diff_mean", "diff_p"))

Events with a secid and a valid window: 133,472  (dropped 0 exiting after OptionMetrics ends on 2025-08-29)
  events with CBOE contracts on the entry day: 84,130
  contract-events: 2,952,565  |  priced entry-to-exit: 2,384,569 (80.8%)
  of which valued at payoff (expired inside the window): 138,968
  retail buy volume priced: 89.1% of 74,807,642 contracts
  procust buy volume priced: 83.8% of 2,216,929 contracts
shape: (20, 12)
┌─────────────┬────────────┬──────────┬─────────────┬──────────┬───────────────┬───────────┬──────────────┬────────────────┬────────┬───────────┬────────┐
│ measure     ┆ era        ┆ retail_n ┆ retail_mean ┆ retail_p ┆ retail_median ┆ procust_n ┆ procust_mean ┆ procust_median ┆ diff_n ┆ diff_mean ┆ diff_p │
│ ---         ┆ ---        ┆ ---      ┆ ---         ┆ ---      ┆ ---           ┆ ---       ┆ ---          ┆ ---            ┆ ---    ┆ ---       ┆ ---    │
│ str         ┆ str        ┆ i64      ┆ f64         ┆ f64      ┆ f64           ┆ i64       ┆ f64       

In [7]:
tbl_otm = option_returns_table(event_option_returns(contracts, moneyness="otm"))

In [2]:
import polars as pl
from analysis.informed_trading import build_event_panel
from analysis.option_returns import build_option_panel, spread_comparison

panel = build_event_panel(verbose=False)
contracts = build_option_panel(panel)   # rebuild: now carries bid/ask at entry and exit

cmp_all = spread_comparison(contracts, measures=("dw_buy", "dw_net"))
cmp_otm = spread_comparison(contracts, measures=("dw_buy", "dw_net"), moneyness="otm")

cols = ["cost_frac", "measure", "era", "retail_mean", "retail_median", "procust_mean",
        "procust_median", "diff_n", "diff_mean", "diff_p"]
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=4, tbl_width_chars=250):
    print("=== all contracts ===");  print(cmp_all.select(cols))
    print("\n=== OTM only ===");     print(cmp_otm.select(cols))

Events with a secid and a valid window: 133,472  (dropped 0 exiting after OptionMetrics ends on 2025-08-29)
  events with CBOE contracts on the entry day: 84,130
  contract-events: 2,952,565  |  priced entry-to-exit: 2,384,569 (80.8%)
  of which valued at payoff (expired inside the window): 138,968
  retail buy volume priced: 89.1% of 74,807,642 contracts
  procust buy volume priced: 83.8% of 2,216,929 contracts
=== all contracts ===
shape: (24, 10)
┌───────────┬─────────┬────────────┬─────────────┬───────────────┬──────────────┬────────────────┬────────┬───────────┬────────┐
│ cost_frac ┆ measure ┆ era        ┆ retail_mean ┆ retail_median ┆ procust_mean ┆ procust_median ┆ diff_n ┆ diff_mean ┆ diff_p │
│ ---       ┆ ---     ┆ ---        ┆ ---         ┆ ---           ┆ ---          ┆ ---            ┆ ---    ┆ ---       ┆ ---    │
│ f64       ┆ str     ┆ str        ┆ f64         ┆ f64           ┆ f64          ┆ f64            ┆ i64    ┆ f64       ┆ f64    │
╞═══════════╪═════════╪═══════

In [3]:
c = contracts.filter(pl.col("ret").is_not_null())
for g in ("retail", "procust"):
    b = pl.col(f"{g}_open_buy") + pl.col(f"{g}_close_buy")
    rel = c.select(((b * (pl.col("ask_entry") - pl.col("bid_entry"))).sum()
                    / (b * pl.col("mid_entry")).sum())).item()
    print(f"{g}: quoted spread = {rel:.1%} of premium, weighted by contracts bought")

retail: quoted spread = 6.8% of premium, weighted by contracts bought
procust: quoted spread = 3.5% of premium, weighted by contracts bought


In [4]:
import polars as pl
from analysis.option_decomposition import (decomposition_panel, event_decomposition,
                                           decomposition_table, right_direction_summary)

d = decomposition_panel()                        # reads the saved contract panel
tbl = decomposition_table(event_decomposition(d))
rd = right_direction_summary(d)
rd_otm = right_direction_summary(d, moneyness="otm")

with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=4, tbl_width_chars=250):
    print(tbl.filter(pl.col("era") == "pooled").select("group", "component", "n", "mean", "se", "median"))
    print(tbl.filter(pl.col("component") == "IV change (vega)").select("group", "era", "mean", "median"))
    cols = ["group", "era", "share_right", "right_but_lost", "iv_outweighed",
            "right_delta_term", "right_vega_term", "right_dP"]
    print("\n=== right-direction positions, all contracts ===");  print(rd.select(cols))
    print("\n=== right-direction positions, OTM only ===");       print(rd_otm.select(cols))

Priced contract-events: 2,384,569  |  decomposable: 1,952,305 (81.9%)
  retail buy volume: 68.8% decomposable, 12.5% expired inside the window, 18.7% missing IV/Greeks
  procust buy volume: 74.1% decomposable, 10.1% expired inside the window, 15.8% missing IV/Greeks
  fit: R^2 of actual on approximated change 0.974; median |residual| = 7.9% of premium
shape: (12, 6)
┌─────────┬───────────────────────┬───────┬─────────┬────────┬─────────┐
│ group   ┆ component             ┆ n     ┆ mean    ┆ se     ┆ median  │
│ ---     ┆ ---                   ┆ ---   ┆ ---     ┆ ---    ┆ ---     │
│ str     ┆ str                   ┆ i64   ┆ f64     ┆ f64    ┆ f64     │
╞═════════╪═══════════════════════╪═══════╪═════════╪════════╪═════════╡
│ retail  ┆ direction (delta)     ┆ 71383 ┆ -0.0397 ┆ 0.0039 ┆ -0.0145 │
│ retail  ┆ convexity (gamma)     ┆ 71383 ┆ 0.3353  ┆ 0.0047 ┆ 0.0995  │
│ retail  ┆ IV change (vega)      ┆ 71383 ┆ -0.1336 ┆ 0.0038 ┆ -0.1719 │
│ retail  ┆ time decay (theta)    ┆ 71383 ┆ -0.